In [1]:
# Install if needed: pip install qdrant-client openai
# IMPORTANT: CAMEL-AI's QdrantStorage expects a default (unnamed) vector, not a named vector
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
import os

load_dotenv()

client = QdrantClient(
    os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True,
    timeout=30.0
)

In [2]:
# Create collection with default (unnamed) vector for CAMEL-AI compatibility
# Note: If you need to fix an existing collection, use: make fix-qdrant-collection
client.create_collection(
    collection_name="log_fixes_2",
    vectors_config=models.VectorParams(
        size=1536,  # text-embedding-3-small dimension
        distance=models.Distance.COSINE
    )
    # Note: Sparse vectors (BM25) can be added later if needed, but CAMEL-AI uses dense vectors
)

True

In [7]:
# Initialize OpenAI embedding model: text-embedding-3-small
from camel.embeddings import OpenAIEmbedding
from camel.types import EmbeddingModelType

# Get OpenAI API credentials (can use separate embedding credentials or fall back to main OpenAI config)
openai_base_url = os.getenv("OPENAI_EMBEDDING_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"))
openai_api_key = os.getenv("OPENAI_EMBEDDING_API_KEY", os.getenv("OPENAI_API_KEY"))

# Initialize OpenAI embedding model: text-embedding-3-small (1536 dimensions)
embedding = OpenAIEmbedding(
    model_type=EmbeddingModelType.TEXT_EMBEDDING_3_SMALL,
    url=openai_base_url,
    api_key=openai_api_key
)

print("OpenAI embedding model initialized: text-embedding-3-small")


OpenAI embedding model initialized: text-embedding-3-small


In [8]:
# Example query strings for testing RAG search functionality
EXAMPLE_QUERIES = [
    "Connection timeout when calling external API",
    "Database connection failed",
    "Bank gateway timeout after 5000ms",
    "Database connection pool exhausted",
    "Failed to authenticate user: Invalid API key",
    "Connection refused: Unable to connect to Redis server",
    "Out of memory: Java heap space",
    "Service unavailable error",
    "Network timeout error",
    "Authentication failed",
    "Database query timeout",
    "Connection pool exhausted",
    "Memory leak detected",
    "API rate limit exceeded",
    "SSL certificate verification failed"
]

print(f"Example queries loaded: {len(EXAMPLE_QUERIES)} queries")
for i, query in enumerate(EXAMPLE_QUERIES[:5], 1):
    print(f"{i}. {query}")


Example queries loaded: 15 queries
1. Connection timeout when calling external API
2. Database connection failed
3. Bank gateway timeout after 5000ms
4. Database connection pool exhausted
5. Failed to authenticate user: Invalid API key


In [9]:
# Example: Generate embedding for a query and search the collection
query_text = EXAMPLE_QUERIES[0]  # Use first example query
print(f"Query: {query_text}")

# Generate embedding vector using OpenAI text-embedding-3-small
query_vector = embedding.embed(query_text)
print(f"Embedding dimension: {len(query_vector)}")

# Search the collection
results = client.search(
    collection_name="log_fixes_2",
    query_vector=query_vector,
    limit=5  # Return top 5 results
)

print(f"\nFound {len(results)} results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(f"   ID: {result.id}")
    if result.payload:
        print(f"   Payload: {result.payload}")


Query: Connection timeout when calling external API
Embedding dimension: 1536


/var/folders/g6/7rp8kcfd12q43tph60swl50m0000gn/T/ipykernel_79716/71390928.py:10: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


ResponseHandlingException: timed out

In [10]:
# Test multiple example queries
for query in EXAMPLE_QUERIES[:3]:  # Test first 3 queries
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    # Generate embedding
    query_vector = embedding.embed(query)
    
    # Search
    results = client.search(
        collection_name="log_fixes_2",
        query_vector=query_vector,
        limit=3
    )
    
    print(f"Top {len(results)} results:")
    for i, result in enumerate(results, 1):
        print(f"  {i}. Score: {result.score:.4f} | ID: {result.id}")



Query: Connection timeout when calling external API


/var/folders/g6/7rp8kcfd12q43tph60swl50m0000gn/T/ipykernel_79716/3309081539.py:11: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


ResponseHandlingException: timed out